# M0 / M1: Popularity Baselines and Final BPR (M1a)

Accepted development stage: popularity diagnostics and BPR trained on all TRAIN interactions; validation labels are held out. M1b and unrelated experiments omitted.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

DATA_DIR = Path(".")

train = pd.read_parquet(DATA_DIR / "train_interactions.parquet")
valid = pd.read_parquet(DATA_DIR / "valid_interactions.parquet")
test = pd.read_parquet(DATA_DIR / "test_interactions.parquet")
metadata = pd.read_parquet(DATA_DIR / "product_metadata.parquet", columns=["parent_asin"])

print("Train:", train.shape)
print("Valid:", valid.shape)
print("Test:", test.shape)
print("Metadata:", metadata.shape)

print("\nUnique users")
print("train:", train["user_id"].nunique())
print("valid:", valid["user_id"].nunique())
print("test :", test["user_id"].nunique())

print("\nUnique items")
print("train:", train["parent_asin"].nunique())
print("valid:", valid["parent_asin"].nunique())
print("test :", test["parent_asin"].nunique())
print("catalog:", metadata["parent_asin"].nunique())

print("\nTargets per user")
print("valid:")
print(valid.groupby("user_id").size().describe())

print("\ntest:")
print(test.groupby("user_id").size().describe())

In [ ]:
train_last = train.groupby("user_id")["timestamp"].max()
valid_time = valid.set_index("user_id")["timestamp"]
test_time = test.set_index("user_id")["timestamp"]

common_users = (
    set(train_last.index)
    & set(valid_time.index)
    & set(test_time.index)
)

common_users = list(common_users)

chronological = (
    (train_last.loc[common_users] < valid_time.loc[common_users])
    & (valid_time.loc[common_users] < test_time.loc[common_users])
)

print("Users checked:", len(common_users))
print("Correct chronological order:", chronological.mean())

In [ ]:
import numpy as np

# -----------------------------------
# Build popularity rankings
# -----------------------------------

all_popularity = (
    train["parent_asin"]
    .value_counts()
    .index
    .tolist()
)

positive_train = train[train["rating"] >= 4]

positive_popularity = (
    positive_train["parent_asin"]
    .value_counts()
    .index
    .tolist()
)


# -----------------------------------
# Build seen-item history
# -----------------------------------

train_seen = (
    train.groupby("user_id")["parent_asin"]
    .agg(set)
    .to_dict()
)

In [ ]:

# -----------------------------------
# Recommendation function
# -----------------------------------

def recommend_popular(user_id, ranked_items, k=10):
    seen = train_seen.get(user_id, set())

    recs = []

    for item in ranked_items:
        if item not in seen:
            recs.append(item)

        if len(recs) == k:
            break

    return recs

In [ ]:
# -----------------------------------
# Evaluation
# -----------------------------------

def evaluate_popularity(valid_df, ranked_items, k=10):
    hits = []
    reciprocal_ranks = []
    ndcgs = []

    for row in valid_df.itertuples(index=False):
        user = row.user_id
        target = row.parent_asin

        recs = recommend_popular(
            user_id=user,
            ranked_items=ranked_items,
            k=k
        )

        if target in recs:
            rank = recs.index(target) + 1

            hits.append(1.0)
            reciprocal_ranks.append(1.0 / rank)
            ndcgs.append(1.0 / np.log2(rank + 1))

        else:
            hits.append(0.0)
            reciprocal_ranks.append(0.0)
            ndcgs.append(0.0)

    return {
        f"HitRate@ {k}": np.mean(hits),
        f"Recall@ {k}": np.mean(hits),
        f"MRR@ {k}": np.mean(reciprocal_ranks),
        f"NDCG@ {k}": np.mean(ndcgs),
    }


# -----------------------------------
# Evaluate both baselines
# -----------------------------------

m0a = evaluate_popularity(
    valid,
    all_popularity,
    k=10
)

m0b = evaluate_popularity(
    valid,
    positive_popularity,
    k=10
)

print("M0a - All interactions")
print(m0a)

print("\nM0b - Rating >= 4")
print(m0b)

In [ ]:
positive_train = train.copy()

In [ ]:
all_users = train["user_id"].unique()

all_items = pd.concat([
    train["parent_asin"],
    valid["parent_asin"],
    test["parent_asin"]
]).unique()

user_to_idx = {
    user: idx
    for idx, user in enumerate(all_users)
}

item_to_idx = {
    item: idx
    for idx, item in enumerate(all_items)
}

idx_to_user = {
    idx: user
    for user, idx in user_to_idx.items()
}

idx_to_item = {
    idx: item
    for item, idx in item_to_idx.items()
}

num_users = len(user_to_idx)
num_items = len(item_to_idx)

print("num_users:", num_users)
print("num_items:", num_items)

In [ ]:
positive_train["user_idx"] = (
    positive_train["user_id"]
    .map(user_to_idx)
)

positive_train["item_idx"] = (
    positive_train["parent_asin"]
    .map(item_to_idx)
)

train_pairs = positive_train[
    ["user_idx", "item_idx"]
].to_numpy()

user_positive_items = (
    positive_train
    .groupby("user_idx")["item_idx"]
    .agg(set)
    .to_dict()
)
# print(train_pairs.shape)
# print(train_pairs[:5])

In [ ]:
train_items = set(train["parent_asin"])
valid_items = set(valid["parent_asin"])
test_items = set(test["parent_asin"])

valid_cold_items = valid_items - train_items
test_cold_items = test_items - train_items

print("Unique train items:", len(train_items))
print("Unique valid items:", len(valid_items))
print("Unique test items:", len(test_items))

print("\nCold validation items:", len(valid_cold_items))
print("Cold test items:", len(test_cold_items))

In [ ]:
valid_cold_mask = ~valid["parent_asin"].isin(train_items)
test_cold_mask = ~test["parent_asin"].isin(train_items)

print(
    "Cold validation targets:",
    valid_cold_mask.sum(),
    "/",
    len(valid)
)

print(
    "Cold validation target rate:",
    valid_cold_mask.mean()
)

print(
    "\nCold test targets:",
    test_cold_mask.sum(),
    "/",
    len(test)
)

print(
    "Cold test target rate:",
    test_cold_mask.mean()
)

In [ ]:
train_with_idx = train.copy()

train_with_idx["user_idx"] = (
    train_with_idx["user_id"]
    .map(user_to_idx)
)

train_with_idx["item_idx"] = (
    train_with_idx["parent_asin"]
    .map(item_to_idx)
)

user_seen_items = (
    train_with_idx
    .groupby("user_idx")["item_idx"]
    .agg(set)
    .to_dict()
)

In [ ]:
import random
import torch
from torch.utils.data import Dataset, DataLoader


class BPRDataset(Dataset):
    def __init__(
        self,
        train_pairs,
        user_seen_items,
        num_items
    ):
        self.train_pairs = train_pairs
        self.user_seen_items = user_seen_items
        self.num_items = num_items

    def __len__(self):
        return len(self.train_pairs)

    def __getitem__(self, idx):
        user_idx, positive_item_idx = self.train_pairs[idx]

        seen = self.user_seen_items[user_idx]

        while True:
            negative_item_idx = random.randrange(self.num_items)

            if negative_item_idx not in seen:
                break

        return (
            torch.tensor(user_idx, dtype=torch.long),
            torch.tensor(positive_item_idx, dtype=torch.long),
            torch.tensor(negative_item_idx, dtype=torch.long),
        )

In [ ]:
dataset = BPRDataset(
    train_pairs=train_pairs,
    user_seen_items=user_seen_items,
    num_items=num_items
)

loader = DataLoader(
    dataset,
    batch_size=2048,
    shuffle=True
)

In [ ]:
import torch.nn as nn
class BPRModel(nn.Module):
    def __init__(
        self,
        num_users,
        num_items,
        embedding_dim=64
    ):
        super().__init__()

        self.user_embedding = nn.Embedding(
            num_users,
            embedding_dim
        )

        self.item_embedding = nn.Embedding(
            num_items,
            embedding_dim
        )

        nn.init.normal_(
            self.user_embedding.weight,
            std=0.01
        )

        nn.init.normal_(
            self.item_embedding.weight,
            std=0.01
        )

    def forward(
        self,
        user_idx,
        positive_item_idx,
        negative_item_idx
    ):
        user_vec = self.user_embedding(user_idx)

        positive_vec = self.item_embedding(
            positive_item_idx
        )

        negative_vec = self.item_embedding(
            negative_item_idx
        )

        positive_score = (
            user_vec * positive_vec
        ).sum(dim=1)

        negative_score = (
            user_vec * negative_vec
        ).sum(dim=1)

        return positive_score, negative_score

In [ ]:
import torch.nn.functional as F


def bpr_loss(
    positive_score,
    negative_score
):
    return -F.logsigmoid(
        positive_score - negative_score
    ).mean()

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

model = BPRModel(
    num_users=num_users,
    num_items=num_items,
    embedding_dim=64
).to(device)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    weight_decay=1e-6
)

In [ ]:
num_epochs = 10

for epoch in range(num_epochs):

    model.train()

    total_loss = 0.0

    for users, positives, negatives in loader:

        users = users.to(device)
        positives = positives.to(device)
        negatives = negatives.to(device)

        optimizer.zero_grad()

        positive_score, negative_score = model(
            users,
            positives,
            negatives
        )

        loss = bpr_loss(
            positive_score,
            negative_score
        )

        loss.backward()

        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(loader)

    print(
        f"Epoch {epoch + 1:02d} | "
        f"Loss: {avg_loss:.4f}"
    )

In [ ]:
torch.save(
    model.state_dict(),
    "bpr_m1a_all_interactions.pt"
)

In [ ]:
train_item_indices = sorted(
    {
        item_to_idx[item]
        for item in train["parent_asin"].unique()
    }
)

print("BPR candidate items:", len(train_item_indices))

In [ ]:
import numpy as np
import torch


@torch.no_grad()
def evaluate_bpr_batched(
    model,
    eval_df,
    user_to_idx,
    item_to_idx,
    user_seen_items,
    train_item_indices,
    k=10,
    batch_size=256
):
    model.eval()

    candidate_tensor = torch.tensor(
        train_item_indices,
        dtype=torch.long,
        device=device
    )

    candidate_embeddings = model.item_embedding(
        candidate_tensor
    )

    candidate_pos = {
        item_idx: pos
        for pos, item_idx in enumerate(train_item_indices)
    }

    hits = []
    reciprocal_ranks = []
    ndcgs = []

    user_ids = eval_df["user_id"].to_numpy()
    target_items = eval_df["parent_asin"].to_numpy()

    for start in range(0, len(eval_df), batch_size):

        end = min(
            start + batch_size,
            len(eval_df)
        )

        batch_users = user_ids[start:end]
        batch_targets = target_items[start:end]

        batch_user_indices = [
            user_to_idx[user]
            for user in batch_users
        ]

        user_tensor = torch.tensor(
            batch_user_indices,
            dtype=torch.long,
            device=device
        )

        user_embeddings = model.user_embedding(
            user_tensor
        )

        # shape:
        # [batch_size, num_candidate_items]
        scores = torch.matmul(
            user_embeddings,
            candidate_embeddings.T
        )

        # Remove items already seen during training
        for row_idx, user_idx in enumerate(batch_user_indices):

            seen = user_seen_items.get(
                user_idx,
                set()
            )

            seen_positions = [
                candidate_pos[item]
                for item in seen
                if item in candidate_pos
            ]

            if seen_positions:

                scores[
                    row_idx,
                    torch.tensor(
                        seen_positions,
                        dtype=torch.long,
                        device=device
                    )
                ] = -torch.inf

        topk_positions = torch.topk(
            scores,
            k=k,
            dim=1
        ).indices.cpu().numpy()

        for row_idx, target_asin in enumerate(batch_targets):

            # target might be cold-start
            target_global_idx = item_to_idx.get(
                target_asin
            )

            if (
                target_global_idx is None
                or target_global_idx not in candidate_pos
            ):
                hits.append(0.0)
                reciprocal_ranks.append(0.0)
                ndcgs.append(0.0)
                continue

            target_position = candidate_pos[
                target_global_idx
            ]

            rec_positions = topk_positions[
                row_idx
            ]

            matches = np.where(
                rec_positions == target_position
            )[0]

            if len(matches) > 0:

                rank = matches[0] + 1

                hits.append(1.0)

                reciprocal_ranks.append(
                    1.0 / rank
                )

                ndcgs.append(
                    1.0 / np.log2(rank + 1)
                )

            else:

                hits.append(0.0)
                reciprocal_ranks.append(0.0)
                ndcgs.append(0.0)

    return {
        f"HitRate@{k}": float(np.mean(hits)),
        f"Recall@{k}": float(np.mean(hits)),
        f"MRR@{k}": float(np.mean(reciprocal_ranks)),
        f"NDCG@{k}": float(np.mean(ndcgs)),
    }

In [ ]:
bpr_valid_metrics = evaluate_bpr_batched(
    model=model,
    eval_df=valid,
    user_to_idx=user_to_idx,
    item_to_idx=item_to_idx,
    user_seen_items=user_seen_items,
    train_item_indices=train_item_indices,
    k=10,
    batch_size=256
)

print("M1a BPR - All Interactions")
print(bpr_valid_metrics)